# 04 - Finance, inventory/suppliers and HSE EDA (SYNTHETIC)

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30); pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
from og_oip import config
NOTICE = config.SYNTHETIC_NOTICE
print(NOTICE)

All operational, production, financial, maintenance, inventory, sensor and HSE data in this project are synthetic/simulated and created for portfolio demonstration purposes. They do not represent actual operations of a real company.


In [2]:
M = {p.stem: pd.read_csv(p, parse_dates=[c for c in ("date","month") if c in pd.read_csv(p, nrows=0).columns]) for p in config.MARTS_DIR.glob("mart_*.csv")}
wd, fm, es = M["mart_well_daily"], M["mart_field_monthly"], M["mart_equipment_summary"]

## Finance

In [3]:
from og_oip.analytics import financial, inventory, hse
fields = pd.read_csv(config.PROCESSED_DIR / "dim_field.csv")
print(financial.kpis(fm)); financial.by_year(fm)

{'revenue_usd_synthetic': 956148995.57, 'operating_cost_usd': 314236614.49, 'operating_margin_usd': 641912381.08, 'maintenance_cost_usd': 9728183.48, 'energy_cost_usd': 14105024.879999999, 'cost_per_bbl_oil_sold': 23.87928811461057, 'cost_per_boe_sold': 20.293140479962613, 'revenue_per_bbl_oil_sold': 72.65912466237485, 'estimated_lost_revenue_usd': 95544789.75505157, 'lost_revenue_pct_of_revenue': 9.99266748150411}


   year    revenue_usd       opex_usd  oil_sold_bbl  lost_revenue_usd  cost_per_bbl  revenue_per_bbl
0  2022 255,766,831.14  89,036,144.85  3,338,085.80     24,456,472.51         26.67            76.62
1  2023 344,972,393.84 110,800,641.14  4,929,572.80     34,783,121.36         22.48            69.98
2  2024 355,409,770.59 114,399,828.50  4,891,720.90     36,305,195.88         23.39            72.66

In [4]:
financial.cost_breakdown(fm)

         category            usd  share_pct
4      Production 124,138,221.24      39.50
1           Labor  79,120,370.43      25.18
5  Transportation  39,473,932.24      12.56
6       Utilities  26,313,492.03       8.37
3           Other  21,357,390.19       6.80
0          Energy  14,105,024.88       4.49
2     Maintenance   9,728,183.48       3.10

## Inventory and suppliers

In [5]:
mi, sp = M["mart_inventory"], M["mart_supplier_performance"]
inv = pd.read_csv(config.PROCESSED_DIR / "fact_inventory.csv", parse_dates=["date"]); dm = pd.read_csv(config.PROCESSED_DIR / "dim_material.csv")
print(inventory.kpis(inv, mi, dm)); inventory.by_category(mi)

{'inventory_value_end_usd': 491061.8, 'avg_inventory_value_usd': 500754.2282349715, 'stockout_rate_pct': 2.139507285735477, 'stockout_days_total': 1875, 'annualised_inventory_turnover': 10.702454379819665, 'avg_days_of_inventory': 34.28028641679957, 'low_stock_items_at_end': 31, 'consumption_value_usd': 16081566.09}


  material_category  stockout_days  item_warehouse_pairs  avg_inventory_value_usd  consumption_value_usd  total_consumption  days  stockout_rate_pct
4        Lubricants            722                    10                65,234.03           3,693,010.71          57,517.00  1096               6.59
1         Chemicals            624                    10                69,795.37           3,874,541.34          42,923.00  1096               5.69
6             Seals            122                    10                20,308.12             752,893.38           4,108.00  1096               1.11
3           Filters            112                    10                17,988.18             718,067.66           4,749.00  1096               1.02
0          Bearings             97                    10                62,801.38           1,703,649.82           1,807.00  1096               0.89
5            Piping             67                    10                80,330.47           1,727,705.99  

In [6]:
sp[["supplier_id","supplier_name","pos_received","on_time_pct","avg_quoted_lead_days","avg_actual_lead_days"]]

  supplier_id                  supplier_name  pos_received  on_time_pct  avg_quoted_lead_days  avg_actual_lead_days
0     SUP-001      Aurelia Industrial Supply           229        61.14                 16.76                 19.22
1     SUP-002         Borealis Machine Parts           248        92.34                 12.35                 12.08
2     SUP-003      Cobalt Valve & Fitting Co           173        87.86                 24.99                 25.34
3     SUP-004  Delmar Lubricants & Chemicals           347        61.38                  6.00                  8.49
4     SUP-005    Everglen Filtration Systems           269        80.30                 11.80                 12.55
5     SUP-006       Fulcrum Electrical Works           225        73.33                 20.77                 22.19
6     SUP-007      Granite Pipe & Flange Ltd           214        83.18                 26.79                 27.25
7     SUP-008         Helix Chemical Trading           228        58.77 

In [7]:
inventory.supplier_association(sp)

{'n_suppliers': 8, 'spearman_rho_ontime_vs_stockout_days': -0.6826469692905576, 'p_value': 0.06209269238231715}

n = 8 suppliers: correlation is indicative only.

## HSE

In [8]:
hm, em = M["mart_hse_monthly"], M["mart_equipment_monthly"]; h = pd.read_csv(config.PROCESSED_DIR / "fact_hse.csv", parse_dates=["date"])
print(hse.kpis(h, hm)); print(hse.maintenance_association(hm, em)); hse.by_field_rate(hm)

{'incident_count': 394, 'lost_time_incidents': 29, 'days_lost': 170, 'portfolio_incident_rate_per_200k_h': 44.791560087764175, 'portfolio_ltir_per_200k_h': 3.296840717119698, 'high_or_critical_share_pct': 17.766497461928935}
{'n_field_months': 144, 'spearman_rho_failures_vs_incidents': -0.03820967738954551, 'p_value': 0.6493351019867414}


    field_id  incidents   lti  exposure_hours  incident_rate_per_200k_h  ltir_per_200k_h
0  FIELD-001     115.00 11.00      552,390.00                     41.64             3.98
1  FIELD-002     108.00  9.00      483,210.00                     44.70             3.73
2  FIELD-003      90.00  5.00      377,460.00                     47.69             2.65
3  FIELD-004      81.00  4.00      346,200.00                     46.79             2.31

In [9]:
hse.distributions(h)["root_cause"]

                 root_cause  incidents
0         Equipment Failure         78
1               Human Error         77
2                   Unknown         73
3    Procedure Not Followed         49
4    Inadequate Maintenance         49
5  Environmental Conditions         35
6       Contractor Activity         33